In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession

# Initialize local SparkSession
spark = (
    SparkSession.builder
    .appName("03_Gold_Star_Schema")
    .getOrCreate()
)

SILVER_PATH = "/home/jovyan/data_lake/silver"
GOLD_PATH = "/home/jovyan/data_lake/gold"
Path(GOLD_PATH).mkdir(parents=True, exist_ok=True)

# Load curated Silver dataset as a temporary SQL view
spark.read.parquet(f"{SILVER_PATH}/clean_wb_indicators").createOrReplaceTempView("clean_wb_indicators")

# 1. Conformed Dimensions
dim_country = spark.createDataFrame(
    [(1, "EGY", "Egypt", "Middle East & North Africa")],
    ["country_key", "country_code", "country_name", "region"]
)
dim_country.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_country")
dim_country.createOrReplaceTempView("dim_country")

dim_source_system = spark.createDataFrame(
    [(1, "EIA_API"), (2, "World_Bank_API"), (3, "IRENA_Dataset")],
    ["source_key", "source_name"]
)
dim_source_system.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_source_system")

dim_year = spark.sql("""
    SELECT DISTINCT
        year AS year_key,
        year,
        CAST(FLOOR(year / 10) * 10 AS INT) AS decade
    FROM clean_wb_indicators
    WHERE year IS NOT NULL
""")
dim_year.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_year")
dim_year.createOrReplaceTempView("dim_year")

# 2. Domain Dimension & Fact Table
dim_indicator = spark.sql("""
    SELECT
        ROW_NUMBER() OVER (ORDER BY indicator_id) AS indicator_key,
        indicator_id,
        indicator_name
    FROM (
        SELECT DISTINCT indicator_id, indicator_name
        FROM clean_wb_indicators
    )
""")
dim_indicator.write.mode("overwrite").parquet(f"{GOLD_PATH}/dim_indicator")
dim_indicator.createOrReplaceTempView("dim_indicator")

fact_wb_indicators = spark.sql("""
    SELECT
        s.year AS year_key,
        1 AS country_key,
        2 AS source_key,
        d.indicator_key,
        s.indicator_value
    FROM clean_wb_indicators s
    JOIN dim_indicator d
        ON s.indicator_id = d.indicator_id
""")
fact_wb_indicators.write.mode("overwrite").parquet(f"{GOLD_PATH}/fact_wb_indicators")
fact_wb_indicators.createOrReplaceTempView("fact_wb_indicators")

# 3. Pre-Aggregated Executive Annual Summary
annual_summary = spark.sql("""
    SELECT
        y.year,
        c.country_name,
        MAX(CASE WHEN i.indicator_id = 'EG.ELC.ACCS.ZS' THEN f.indicator_value END) AS electricity_access_pct,
        MAX(CASE WHEN i.indicator_id = 'EG.FEC.RNEW.ZS' THEN f.indicator_value END) AS renewable_consumption_pct,
        MAX(CASE WHEN i.indicator_id = 'EG.USE.ELEC.KH.PC' THEN f.indicator_value END) AS electric_consumption_kwh_pc
    FROM fact_wb_indicators f
    JOIN dim_year y ON f.year_key = y.year_key
    JOIN dim_country c ON f.country_key = c.country_key
    JOIN dim_indicator i ON f.indicator_key = i.indicator_key
    GROUP BY y.year, c.country_name
    ORDER BY y.year DESC
""")
annual_summary.write.mode("overwrite").parquet(f"{GOLD_PATH}/vw_egypt_energy_annual_summary")

print("=== Gold Executive Annual Summary (Top 15 Years) ===")
annual_summary.show(15, truncate=False)

spark.stop()